# LangChain: Embedding Concepts

## Outline
* مفهوم Embedding (بردارسازی معنایی)
* راه‌اندازی `OpenAIEmbeddings` در LangChain
* تفاوت متدهای `embed_query` و `embed_documents`
* محاسبه دستی شباهت معنایی (Cosine Similarity) بدون نیاز به Vector Store برای درک ریاضیات پشت پرده


In [1]:
import os
from dotenv import load_dotenv, find_dotenv
_ = load_dotenv(find_dotenv())

## ۱. مفهوم Embedding

در الگوهای پردازش زبان طبیعی مدرن، کلمات و جملات به آرایه‌ای از اعداد (Vector) تبدیل می‌شوند که بار معنایی آن‌ها را در یک فضای چندبعدی مشخص می‌کند. متون هم‌معنی در این فضا به یکدیگر نزدیک‌تر خواهند بود.


In [12]:
# pip install langchain-openai numpy
from langchain_openai import OpenAIEmbeddings
from sklearn.metrics.pairwise import cosine_similarity


# Initialize the embedding model using LangChain
embeddings = OpenAIEmbeddings(model="text-embedding-3-large")
print("Embedding model initialized successfully!")


Embedding model initialized successfully!


## 2. متد `embed_documents`

از این متد برای تبدیل **لیستی از مستندات** (محتوای پایگاه داده کاتالوگ شما) به بردار استفاده می‌شود. خروجی آن لیستی از لیست‌ها خواهد بود.


In [15]:
# A small corpus of documents to teach similarity
documents = [
    "هوش مصنوعی در حال تغییر جهان است.",
    "فناوری هوش مصنوعی زندگی انسان‌ها را دگرگون می‌کند.",
    "رانندگی علی خوب است",
    "هوا کثیف است",
    "هوا آلوده است"
]

# Convert multiple documents into embedding vectors
doc_vectors = embeddings.embed_documents(documents)

print(f"Total documents embedded: {len(doc_vectors)}")
print(f"Dimensions of first document: {len(doc_vectors[0])}")


Total documents embedded: 5
Dimensions of first document: 3072


In [19]:
cos_sim_matrix = cosine_similarity(doc_vectors)
print(cos_sim_matrix)

[[1.         0.71624472 0.3140736  0.39887742 0.41243253]
 [0.71624472 1.         0.32755316 0.35600791 0.34045087]
 [0.3140736  0.32755316 1.         0.4398166  0.39953402]
 [0.39887742 0.35600791 0.4398166  1.         0.77260347]
 [0.41243253 0.34045087 0.39953402 0.77260347 1.        ]]


## 3. محاسبه دستی شباهت کسینوسی (Cosine Similarity)

برای اینکه درک کنیم دیتابیس‌های برداری (مثل FAISS) چطور کار می‌کنند، بیایید بردار سوال کاربر را با بردار مستندات به صورت ریاضی با فرمول Cosine Similarity مقایسه کنیم.


In [21]:
import numpy as np

def cosine_similarity(vec1, vec2):
    """Calculate cosine similarity between two vectors manually."""
    v1 = np.array(vec1)
    v2 = np.array(vec2)
    dot_product = np.dot(v1, v2)
    norm_v1 = np.linalg.norm(v1)
    norm_v2 = np.linalg.norm(v2)
    return dot_product / (norm_v1 * norm_v2)

text = "یادگیری ماشین و یادگیری عمیق"
query_vector = embeddings.embed_query(text)
print(f"Query: '{text}'\n")
print("Manual Search Results via Math:")

# Compare query vector against all document vectors
for i, doc_vec in enumerate(doc_vectors):
    score = cosine_similarity(query_vector, doc_vec)
    print(f"  Document {i+1}: \"{documents[i]}\"")
    print(f"  Similarity Score: {score * 100:.2f}%\n")


Query: 'یادگیری ماشین و یادگیری عمیق'

Manual Search Results via Math:
  Document 1: "هوش مصنوعی در حال تغییر جهان است."
  Similarity Score: 44.77%

  Document 2: "فناوری هوش مصنوعی زندگی انسان‌ها را دگرگون می‌کند."
  Similarity Score: 47.00%

  Document 3: "رانندگی علی خوب است"
  Similarity Score: 35.46%

  Document 4: "هوا کثیف است"
  Similarity Score: 31.31%

  Document 5: "هوا آلوده است"
  Similarity Score: 27.26%



In [23]:
import numpy as np

def cosine_similarity(vec1, vec2):
    """Calculate cosine similarity between two vectors manually."""
    v1 = np.array(vec1)
    v2 = np.array(vec2)
    dot_product = np.dot(v1, v2)
    norm_v1 = np.linalg.norm(v1)
    norm_v2 = np.linalg.norm(v2)
    return dot_product / (norm_v1 * norm_v2)

text = "there is an air polution"
query_vector = embeddings.embed_query(text)
print(f"Query: '{text}'\n")
print("Manual Search Results via Math:")

# Compare query vector against all document vectors
for i, doc_vec in enumerate(doc_vectors):
    score = cosine_similarity(query_vector, doc_vec)
    print(f"  Document {i+1}: \"{documents[i]}\"")
    print(f"  Similarity Score: {score * 100:.2f}%\n")


Query: 'there is an air polution'

Manual Search Results via Math:
  Document 1: "هوش مصنوعی در حال تغییر جهان است."
  Similarity Score: 12.39%

  Document 2: "فناوری هوش مصنوعی زندگی انسان‌ها را دگرگون می‌کند."
  Similarity Score: 9.46%

  Document 3: "رانندگی علی خوب است"
  Similarity Score: 10.61%

  Document 4: "هوا کثیف است"
  Similarity Score: 37.41%

  Document 5: "هوا آلوده است"
  Similarity Score: 51.37%

